In [3]:
from torchvision import datasets, transforms
from PIL import Image
import shutil
import os

def generate_mnist_samples(number: int, max_samples: int = 100, test_fraction: float = 0.2, output_dir: str = "../../tests/generated_samples") -> None:
    """
    Generate and save MNIST samples for a specified number, split into train and test sets.

    Args:
        number (int): The MNIST digit to generate samples for (0-9).
        max_samples (int, optional): The maximum number of samples to generate. Defaults to 100.
        test_fraction (float, optional): Fraction of samples to use for test set. Defaults to 0.2.
        output_dir (str, optional): The base output directory. Defaults to "../../tests/generated_samples".

    Returns:
        None
    """
    # Set up the output directories
    digit_output_dir = os.path.join(output_dir, f"mnist_{number}")
    train_dir = os.path.join(digit_output_dir, "train")
    test_dir = os.path.join(digit_output_dir, "test")
    # Remove existing directories with files inside
    if os.path.exists(digit_output_dir):
        shutil.rmtree(digit_output_dir)
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    # Download and load MNIST dataset
    transform = transforms.Compose([transforms.ToTensor()])
    mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

    # Filter for only the specified number
    filtered_dataset = [(img, label) for img, label in mnist_train if label == number][:max_samples]
    
    # Calculate split indices
    test_size = int(len(filtered_dataset) * test_fraction)
    train_size = len(filtered_dataset) - test_size

    # Split dataset into train and test
    train_dataset = filtered_dataset[:train_size]
    test_dataset = filtered_dataset[train_size:]

    # Generate and save training images
    for i, (img, _) in enumerate(train_dataset):
        pil_img = transforms.ToPILImage()(img.squeeze())
        pil_img = pil_img.resize((100, 100), Image.BILINEAR)
        pil_img.save(os.path.join(train_dir, f"mnist_{number}_{i:05d}.png"))

    # Generate and save test images with separate counter
    for i, (img, _) in enumerate(test_dataset):
        pil_img = transforms.ToPILImage()(img.squeeze())
        pil_img = pil_img.resize((100, 100), Image.BILINEAR)
        pil_img.save(os.path.join(test_dir, f"mnist_{number}_{i:05d}.png"))

    print(f"Generated {train_size} training images and {test_size} test images of the number {number}")
    print(f"Training images in: {train_dir}")
    print(f"Test images in: {test_dir}")

In [4]:
def wait_for_kafka_idle(topic: str, idle_timeout: int = 30, bootstrap_servers: str = "localhost:29092") -> None:
    """
    Wait until a Kafka topic has been idle (no new messages) for the specified duration.
    
    Args:
        topic (str): Name of the Kafka topic to monitor
        idle_timeout (int, optional): Time in seconds to wait for no activity before considering idle. Defaults to 30.
        bootstrap_servers (str, optional): Kafka bootstrap servers. Defaults to "localhost:29092".
        
    Returns:
        None
    """
    from kafka import KafkaConsumer
    import time
    
    # Create consumer
    consumer = KafkaConsumer(
        topic,
        bootstrap_servers=bootstrap_servers,
        auto_offset_reset='latest',
        enable_auto_commit=True,
        group_id=None,
        consumer_timeout_ms=1000  # 1 second timeout for poll()
    )
    
    try:
        last_message_time = time.time()
        print(f"Monitoring topic {topic} for {idle_timeout} seconds of inactivity...")
        
        while True:
            # Try to get message
            messages = consumer.poll(timeout_ms=1000)
            current_time = time.time()
            
            if messages:
                # Reset timer if we got messages
                last_message_time = current_time
                print("Messages received, resetting idle timer...")
            else:
                # Check if we've been idle long enough
                idle_duration = current_time - last_message_time
                if idle_duration >= idle_timeout:
                    print(f"No messages received for {idle_timeout} seconds. Topic {topic} is idle.")
                    return
                
                if idle_duration >= 5:  # Only print every 5 seconds
                    print(f"No messages for {int(idle_duration)} seconds...")
    
    finally:
        consumer.close()



In [5]:
def train_mnist(class_number: int, samples: int):
    generate_mnist_samples(class_number, max_samples=samples)
    !cd ../../ && make train_mnist_{class_number}
    wait_for_kafka_idle(topic="contour-analysis-output-topic", idle_timeout=10, bootstrap_servers="localhost:29092")
    !cd ../../ && make post_process {class_number} mnist-{class_number}


In [6]:
from neo4j import GraphDatabase

def clean_neo4j_db() -> None:
    """Cleans all nodes and relationships from Neo4j database"""
    uri = "bolt://localhost:7687"
    user = "neo4j"
    password = "111122223333"

    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session() as session:
        # Delete all nodes and relationships
        session.run("MATCH (n) DETACH DELETE n")
    driver.close()


def clean_kafka_topics() -> None:
    """Deletes all messages from Kafka topics by recreating them"""
    from kafka.admin import KafkaAdminClient, NewTopic
    from kafka.errors import TopicAlreadyExistsError, UnknownTopicOrPartitionError
    
    topics = [
        "connector-output-topic",
        "line-detector-output-topic", 
        "angle-point-detector-output-topic",
        "skeletonization-output-topic",
        "contour-analysis-output-topic",
        "classification-output-topic",
        "dlq-topic"
    ]
    
    admin_client = KafkaAdminClient(bootstrap_servers="localhost:29092")
    
    # Delete existing topics
    for topic in topics:
        try:
            admin_client.delete_topics([topic])
            logging.info(f"Deleted topic: {topic}")
        except UnknownTopicOrPartitionError:
            logging.info(f"Topic {topic} does not exist")
    
    time.sleep(5)  # Wait for topics to be fully deleted
    
    # Recreate topics
    topic_list = []
    for topic in topics:
        topic_list.append(NewTopic(
            name=topic,
            num_partitions=1,
            replication_factor=1
        ))
    
    for topic in topic_list:
        try:
            admin_client.create_topics([topic])
            logging.info(f"Created topic: {topic.name}")
        except TopicAlreadyExistsError:
            logging.warning(f"Topic {topic.name} already exists")
    
    admin_client.close()


In [7]:
import os
import logging
from kafka import KafkaConsumer
import json
import time
from typing import Dict, Any, List, Tuple
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
import pandas as pd
from datetime import datetime

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def save_confusion_matrix(cm: np.ndarray, classes: List[str], timestamp: str) -> None:
    """Plot and save confusion matrix."""
    plt.figure(figsize=(10,7))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix')
    plt.colorbar()
    
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    
    # Add text annotations
    thresh = cm.max() / 2.
    for i, j in np.ndindex(cm.shape):
        plt.text(j, i, format(cm[i, j], 'd'),
                horizontalalignment="center",
                color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    
    # Create metrics directory if it doesn't exist
    metrics_dir = "../../training_results/metrics"
    os.makedirs(metrics_dir, exist_ok=True)
    
    # Save plot
    plt.savefig(f"{metrics_dir}/confusion_matrix_{timestamp}.png")
    plt.close()

def test_mnist_all(classes: List[int]) -> Tuple[Dict[str, Any], List[str], List[str]]:
    """Test MNIST classification for all classes and calculate overall metrics.
    
    Args:
        classes: List of class numbers to test
        
    Returns:
        Tuple containing results dict, true labels and predicted labels
    """
    all_results: Dict[str, Any] = {}
    all_y_true: List[str] = []
    all_y_pred: List[str] = []
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    for class_number in classes:
        test_folder = f"../../tests/generated_samples/mnist_{class_number}/test"
        test_images = [f for f in os.listdir(test_folder) if f.endswith((".png", ".jpg", ".jpeg"))]
        nuclio_volume_path = f"/opt/nuclio/shared_storage/mnist_{class_number}/test"
        test_images = [os.path.join(nuclio_volume_path, f) for f in test_images]

        consumer = KafkaConsumer(
            "classification-output-topic", "dlq-topic",
            bootstrap_servers="localhost:29092",
            auto_offset_reset="latest",
            enable_auto_commit=True,
            value_deserializer=lambda x: json.loads(x.decode("utf-8")),
            group_id=f"test-mnist-{class_number}",
            consumer_timeout_ms=1000
        )

        for image_file in test_images:
            logger.debug(f"Processing image: {image_file}")
            os.system(f"cd ../../ && make classify {image_file}")
            
            start_time = time.time()
            dlq_result = False
            expected_name = f"mnist-{class_number}"
            all_y_true.append(expected_name)
            
            while time.time() - start_time < 10:
                try:
                    messages = consumer.poll(timeout_ms=1000)
                    for topic_partition, msgs in messages.items():
                        for msg in msgs:
                            if msg.topic == "dlq-topic":
                                dlq_result = True
                                logger.warning(f"Image {image_file} failed and went to DLQ")
                                all_y_pred.append("error")
                                break
                            elif msg.topic == "classification-output-topic":
                                result = msg.value
                                
                                # Verify the result corresponds to the current image
                                if result['image_path'].split('/')[-1] != image_file.split('/')[-1]:
                                    logger.warning(f"Received result for different image. Expected {image_file}, got {result['parameters']['image_path']}")
                                    continue
                                    
                                all_results[image_file] = result
                                
                                if 'classification_results' in result:
                                    top_result = sorted(
                                        result['classification_results'],
                                        key=lambda x: x['combined_score'],
                                        reverse=True
                                    )[0]
                                    predicted_name = top_result['concept_name']
                                    all_y_pred.append(predicted_name)
                                    
                                    logger.debug(
                                        f"Image {image_file}: {'Correct' if predicted_name == expected_name else 'Incorrect'} "
                                        f"classification - got {predicted_name}, expected {expected_name}"
                                    )
                                break
                    if dlq_result or image_file in all_results:
                        break
                except Exception as e:
                    logger.error(f"Error reading from Kafka: {e}")
                    time.sleep(0.1)
                    continue

            if time.time() - start_time >= 10:
                logger.warning(f"Timeout waiting for classification of {image_file}")
                all_results[image_file] = "timeout"
                all_y_pred.append("timeout")

        consumer.close()

    # Calculate overall metrics
    total = len(all_y_true)
    successful = sum(1 for r in all_results.values() if r != "timeout")
    failed_dlq = total - len(all_results)
    actual_total = total - failed_dlq
    
    metrics_dir = "training_results/metrics"
    os.makedirs(metrics_dir, exist_ok=True)
    
    if actual_total > 0:
        # Calculate precision, recall, F1 score
        labels = sorted(list(set(all_y_true + all_y_pred)))
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_y_true, all_y_pred, labels=labels, average='weighted'
        )
        accuracy = accuracy_score(all_y_true, all_y_pred)
        
        # Save metrics to CSV
        metrics_df = pd.DataFrame({
            'Metric': ['Total Images', 'Failed (DLQ)', 'Successfully Classified', 
                      'Success Rate (%)', 'Accuracy (%)', 'Precision (%)', 
                      'Recall (%)', 'F1 Score (%)'],
            'Value': [total, failed_dlq, successful, 
                     (successful/actual_total)*100, accuracy*100,
                     precision*100, recall*100, f1*100]
        })
        metrics_df.to_csv(f"{metrics_dir}/metrics_{timestamp}.csv", index=False)
        
        logger.info("\nOverall Classification Metrics:")
        logger.info(f"Total images across all classes: {total}")
        logger.info(f"Failed (DLQ): {failed_dlq}")
        logger.info(f"Successfully classified: {successful}")
        logger.info(f"Overall success rate: {(successful/actual_total)*100:.2f}%")
        logger.info(f"Overall accuracy: {accuracy*100:.2f}%")
        logger.info(f"Overall precision: {precision*100:.2f}%")
        logger.info(f"Overall recall: {recall*100:.2f}%")
        logger.info(f"Overall F1 Score: {f1*100:.2f}%")
        
        # Generate and save confusion matrix
        cm = confusion_matrix(all_y_true, all_y_pred, labels=labels)
        save_confusion_matrix(cm, labels, timestamp)
    else:
        logger.warning("No successful classifications to calculate metrics")

    return all_results, all_y_true, all_y_pred

In [ ]:
train_set_sizes = [20, 30, 40, 50, 100, 200, 500]
classes = [1, 2, 3, 4, 5, 6, 7]

for size in train_set_sizes:
    clean_neo4j_db()
    clean_kafka_topics()
    for class_number in classes:
        train_mnist(class_number, size)
    test_mnist_all(classes)